In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)
print("NIA notebook is working!")

Pandas version: 3.0.5
NIA notebook is working!


In [2]:
import pandas as pd

DATA_PATH = "../data/raw/archive/twcs/twcs.csv"

df = pd.read_csv(DATA_PATH, nrows=10)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (10, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tweet_id                 10 non-null     int64  
 1   author_id                10 non-null     str    
 2   inbound                  10 non-null     bool   
 3   created_at               10 non-null     str    
 4   text                     10 non-null     str    
 5   response_tweet_id        8 non-null      str    
 6   in_response_to_tweet_id  9 non-null      float64
dtypes: bool(1), float64(1), int64(1), str(4)
memory usage: 622.0 bytes


In [4]:
df["inbound"].value_counts()

inbound
False    5
True     5
Name: count, dtype: int64

In [5]:
df[["inbound", "text"]]

,inbound,text
0,False,@115712 I understand. I would like to assist y...
1,True,@sprintcare and how do you propose we do that
2,True,@sprintcare I have sent several private messag...
3,False,@115712 Please send us a Private Message so th...
4,True,@sprintcare I did.
5,False,@115712 Can you please send us a private messa...
6,True,@sprintcare is the worst customer service
7,False,@115713 This is saddening to hear. Please shoo...
8,True,@sprintcare You gonna magically change your co...
9,False,@115713 We understand your concerns and we'd l...


In [6]:
from collections import Counter

brand_counts = Counter()

for chunk in pd.read_csv(DATA_PATH, usecols=["author_id", "inbound"], chunksize=100_000):
    support_accounts = chunk.loc[chunk["inbound"] == False, "author_id"]
    brand_counts.update(support_accounts)

brand_counts.most_common(20)

[('AmazonHelp', 169840),
 ('AppleSupport', 106860),
 ('Uber_Support', 56270),
 ('SpotifyCares', 43265),
 ('Delta', 42253),
 ('Tesco', 38573),
 ('AmericanAir', 36764),
 ('TMobileHelp', 34317),
 ('comcastcares', 33031),
 ('British_Airways', 29361),
 ('SouthwestAir', 28977),
 ('VirginTrains', 27817),
 ('Ask_Spectrum', 25860),
 ('XboxSupport', 24557),
 ('sprintcare', 22381),
 ('hulu_support', 21872),
 ('sainsburys', 19466),
 ('GWRHelp', 19364),
 ('AskPlayStation', 19098),
 ('ChipotleTweets', 18749)]

In [7]:
spotify_chunks = []

for chunk in pd.read_csv(DATA_PATH, chunksize=100_000):
    spotify_rows = chunk[
        (chunk["author_id"] == "SpotifyCares") |
        (
            (chunk["inbound"] == True) &
            (chunk["text"].str.contains("@SpotifyCares", case=False, na=False))
        )
    ]
    spotify_chunks.append(spotify_rows)

spotify_df = pd.concat(spotify_chunks, ignore_index=True)

print("Spotify rows:", len(spotify_df))
spotify_df.head()

Spotify rows: 74618


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,848,SpotifyCares,False,Tue Oct 31 22:28:16 +0000 2017,@115887 Hmm. Can you try restarting your devic...,849,850.0
1,849,115887,True,Tue Oct 31 23:36:20 +0000 2017,@SpotifyCares doesn’t work and i even tried de...,851,848.0
2,851,SpotifyCares,False,Tue Oct 31 23:39:03 +0000 2017,@115887 Could you send us a DM with your accou...,NaN,849.0
3,850,115887,True,Tue Oct 31 21:41:37 +0000 2017,@SpotifyCares Premium &amp; when i️ have it on...,848,852.0
4,852,SpotifyCares,False,Tue Oct 31 21:04:13 +0000 2017,"@115887 Thanks. Just to be sure, are you Free ...",850,853.0


In [8]:
print("Spotify rows:", len(spotify_df))

Spotify rows: 74618


In [9]:
spotify_df["inbound"].value_counts()

inbound
False    43265
True     31353
Name: count, dtype: int64

In [10]:
spotify_df["created_at"] = pd.to_datetime(spotify_df["created_at"])

print("First tweet:", spotify_df["created_at"].min())
print("Last tweet:", spotify_df["created_at"].max())

C:\Users\ARBAZ SALAM\AppData\Local\Temp\ipykernel_15284\3520734973.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  spotify_df["created_at"] = pd.to_datetime(spotify_df["created_at"])


First tweet: 2013-09-18 19:23:17+00:00
Last tweet: 2017-12-03 22:56:04+00:00


In [11]:
spotify_df["year"] = spotify_df["created_at"].dt.year

spotify_df["year"].value_counts().sort_index()

year
2013        3
2015        6
2016       17
2017    74592
Name: count, dtype: int64

In [12]:
customer_tweets = spotify_df[spotify_df["inbound"] == True].copy()

print("Customer tweets:", len(customer_tweets))

customer_tweets[["text"]].head(20)

Customer tweets: 31353


,text
1,@SpotifyCares doesn’t work and i even tried de...
3,@SpotifyCares Premium &amp; when i️ have it on...
5,@SpotifyCares iphone 7+ and i have the most re...
8,"@SpotifyCares Yes, multiple times. No changes...."
10,@SpotifyCares 2/2... and there is no way to ma...
12,@SpotifyCares @115890 Groove Music quits &amp;...
14,@SpotifyCares ok thx
16,is there a way to find non-explicit songs that...
18,@SpotifyCares I tried it on web browser and it...
19,"@SpotifyCares Desktop app still not working, b..."


In [13]:
sample_customers = customer_tweets[["tweet_id", "text"]].sample(
    n=30,
    random_state=42
)

sample_customers

,tweet_id,text
29133,1301944,@SpotifyCares Best response ever! Me @ the use...
30437,1380867,@SpotifyCares Spotify Student + Hulu renews to...
39027,1790160,@SpotifyCares I just dumped the whole thing. I...
40121,1814781,"@SpotifyCares my band's ""related artists"" sect..."
5043,254988,"@SpotifyCares Uh no....that""s not an answer. G..."
27652,1244355,@SpotifyCares Ummm...so it still says Would in...
23021,1049686,@SpotifyCares They don't have an account so it...
24800,1124160,"@SpotifyCares Yep! And individual songs, are f..."
199,9358,@SpotifyCares I'm new at this how do i DELETE ...
21262,967401,@SpotifyCares But I have done the reset functi...


In [14]:
spotify_df[spotify_df["tweet_id"] == 40476][
    ["tweet_id", "author_id", "inbound", "text", "in_response_to_tweet_id"]
]

,tweet_id,author_id,inbound,text,in_response_to_tweet_id


In [15]:
spotify_df[spotify_df["tweet_id"] == 1824347][
    ["tweet_id", "author_id", "inbound", "text", "in_response_to_tweet_id"]
]

,tweet_id,author_id,inbound,text,in_response_to_tweet_id
40476,1824347,547643,True,@SpotifyCares DM sent,1824346.0


In [16]:
spotify_df[spotify_df["tweet_id"] == 1824346][
    ["tweet_id", "author_id", "inbound", "text", "in_response_to_tweet_id"]
]

,tweet_id,author_id,inbound,text,in_response_to_tweet_id
40475,1824346,SpotifyCares,False,@547643 Hey Romelio! Can you DM us your accoun...,1824348.0


In [17]:
spotify_df[spotify_df["tweet_id"] == 1824348][
    ["tweet_id", "author_id", "inbound", "text", "in_response_to_tweet_id"]
]

,tweet_id,author_id,inbound,text,in_response_to_tweet_id
40477,1824348,547643,True,@SpotifyCares hello. My family gets kicked out...,NaN


In [18]:
conversation_ids = [1824348, 1824346, 1824347]

spotify_df[spotify_df["tweet_id"].isin(conversation_ids)][
    ["tweet_id", "author_id", "inbound", "text"]
].sort_values("tweet_id")

,tweet_id,author_id,inbound,text
40475,1824346,SpotifyCares,False,@547643 Hey Romelio! Can you DM us your accoun...
40476,1824347,547643,True,@SpotifyCares DM sent
40477,1824348,547643,True,@SpotifyCares hello. My family gets kicked out...


In [19]:
pd.set_option("display.max_colwidth", None)

spotify_df[spotify_df["tweet_id"].isin(conversation_ids)][
    ["tweet_id", "author_id", "inbound", "text"]
].sort_values("tweet_id")

,tweet_id,author_id,inbound,text
40475,1824346,SpotifyCares,False,"@547643 Hey Romelio! Can you DM us your account's email address or username, along with your family members'? We'll take a look backstage /RH https://t.co/ldFdZRiNAt"
40476,1824347,547643,True,@SpotifyCares DM sent
40477,1824348,547643,True,@SpotifyCares hello. My family gets kicked out of premium for family because I can't update my address.


## Initial Conversation Insight

A Spotify support case can span multiple tweets and include short
follow-up messages such as "DM sent".

Example:

Customer:
"My family gets kicked out of premium for family because I can't update my address."

SpotifyCares:
"Hey Romelio! Can you DM us your account's email address or username,
along with your family members'? We'll take a look backstage/RH"

Customer:
"DM sent"

This demonstrates that individual tweets should not always be treated
as independent support cases. Conversation context is important for
intent classification, reply generation, and escalation decisions.